In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from xgboost import XGBClassifier

import import_ipynb
from notebooks import DataPreProcessing as dp

In [ ]:
# Get preprocessed data from DataPreProcessing notebook
X_train = dp.X_train
X_test = dp.X_test
y_train = dp.y_train
y_test = dp.y_test
clean_text = dp.clean_text

In [ ]:
# ---------- PATHS ----------
SAVE_DIR = os.path.join(os.getcwd(), 'models')
SAVE_PATH = os.path.join(SAVE_DIR, 'xgboost.pkl')

In [ ]:
# ---------- SAVE / LOAD ----------
def _get_save_path(dataset_choice='SMS'):
    dataset_key = dataset_choice.lower()
    return os.path.join(SAVE_DIR, f'{dataset_key}_xgboost.pkl')

def save_model(model, vectorizer, dataset_choice='SMS'):
    os.makedirs(SAVE_DIR, exist_ok=True)
    save_path = _get_save_path(dataset_choice)
    with open(save_path, 'wb') as f:
        pickle.dump({'model': model, 'vectorizer': vectorizer}, f)
    print(f'Model saved to {save_path}')

def load_model(dataset_choice='SMS'):
    save_path = _get_save_path(dataset_choice)
    with open(save_path, 'rb') as f:
        data = pickle.load(f)
    return data['model'], data['vectorizer']

def has_saved_model(dataset_choice='SMS'):
    return os.path.exists(_get_save_path(dataset_choice))

In [ ]:
# ---------- TRAIN ----------
def train(X_train, y_train, X_test, y_test, dataset_choice='SMS'):
    vectorizer = CountVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = XGBClassifier(n_estimators=100, max_depth=6, random_state=42, eval_metric='logloss')
    model.fit(X_train_vec, y_train, eval_set=[(X_train_vec, y_train), (X_test_vec, y_test)], verbose=False)

    # Extract per-round metrics from XGBoost's eval results
    results = model.evals_result()
    train_losses = results['validation_0']['logloss']
    test_losses = results['validation_1']['logloss']

    # Compute per-round accuracies
    from sklearn.metrics import accuracy_score as acc_score
    train_accuracies = []
    test_accuracies = []
    for i in range(1, model.n_estimators + 1):
        model.n_estimators = i
        # We use the staged approach: predict with first i trees via iteration_range
        pass

    # Simpler approach: just report final accuracy + full loss curves
    y_pred_train = model.predict(X_train_vec)
    y_pred_test = model.predict(X_test_vec)
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)

    print(f'Train Accuracy: {train_acc*100:.2f}%')
    print(f'Test Accuracy: {test_acc*100:.2f}%')
    print(classification_report(y_test, y_pred_test, target_names=['Ham', 'Spam']))

    save_model(model, vectorizer, dataset_choice)
    return model, vectorizer, train_losses, test_losses, train_acc, test_acc

In [ ]:
# ---------- PREDICT ----------
def predict_message(message, model=None, vectorizer=None):
    """Predict whether a message is Spam or Ham.
    If model/vectorizer are not passed, loads the saved model automatically."""
    if model is None or vectorizer is None:
        model, vectorizer = load_model()
    cleaned = clean_text(message)
    msg_vec = vectorizer.transform([cleaned])
    probabilities = model.predict_proba(msg_vec)[0]
    prediction = model.predict(msg_vec)[0]
    label = 'Spam' if prediction == 1 else 'Ham'
    confidence = max(probabilities) * 100
    return label, confidence